# Evaluate
データセットでエージェントを性能評価。  
定時ジョブなどで実行する事でエージェントの性能を定期的に計測する用途を想定。

# Prepare

In [1]:
import mlflow
import zoneinfo

tz_info = zoneinfo.ZoneInfo("Asia/Tokyo")
mlflow.set_experiment("agent-rag")

<Experiment: artifact_location='mlflow-artifacts:/5', creation_time=1767655484458, experiment_id='5', last_update_time=1767655484458, lifecycle_stage='active', name='agent-rag', tags={'mlflow.experimentKind': 'genai_development'}, workspace='default'>

# Define Criteria

In [2]:
# テストデータを作成
eval_dataset = [
    {
        "inputs": {
            "messages": [
                {"role": "user", "content": "東京都の明日の天気を教えてください。"}
            ]
        },
        "expectations": {"expected_response": "東京都の明日の天気は晴れです"},
    },
    {
        "inputs": {
            "messages": [
                {"role": "user", "content": "横浜市の明日の天気を教えてください。"}
            ]
        },
        "expectations": {"expected_response": "横浜市の明日の天気は晴れです"},
    },
    {
        "inputs": {
            "messages": [
                {"role": "user", "content": "群馬県の明日の天気を教えてください。"}
            ]
        },
        "expectations": {"expected_response": "群馬県の天気は豪雨です。"},
    },
]

# Package & Evaluate

In [10]:
import datetime
from agent_assistant import agent, evaluate

ymd = datetime.datetime.now(tz=tz_info).strftime("%Y%m%d_%H%M%S")
with mlflow.start_run(run_name=f"pkg_{ymd}"):
    # モデルを記録
    res = mlflow.pyfunc.log_model(
        python_model="../src/agent_assistant/agent.py", code_paths=["../src/"], name=f"{agent.AGENT_NAME}_{ymd}"
    )
    logged_model = mlflow.pyfunc.load_model(res.model_uri)

    # モデルを評価
    mlflow.set_active_model(model_id=res.model_id)
    res = evaluate.eval_responses(logged_model, eval_dataset)
    print(res)

2026/03/07 15:01:12 INFO mlflow.pyfunc: Predicting on input example to validate output
2026/03/07 15:01:12 WARNING mlflow.tracing.fluent: Failed to start span predict_stream: 'NonRecordingSpan' object has no attribute 'context'. For full traceback, set logging level to debug.
2026/03/07 15:01:21 WARNING mlflow.agent_assistant.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/03/07 15:01:21 WARNING mlflow.tracing.fluent: Failed to start span predict_stream: 'NonRecordingSpan' object has no attribute 'context'. For full traceback, set logging level to debug.
2026/03/07 15:01:22 INFO mlflow.models.model: Found the following environment variables used during model inference: [CONTEXT7_API_KEY, ENV_GEMINI_API_KEY]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.


2026/03/07 15:01:24 INFO mlflow.tracking.fluent: Active model is set to the logged model with ID: m-c08d2d26de7345c1a5e3de80893b5033
2026/03/07 15:01:24 INFO mlflow.genai.agent_assistant.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.
2026/03/07 15:01:24 WARNING mlflow.tracing.fluent: Failed to start span predict_stream: 'NonRecordingSpan' object has no attribute 'context'. For full traceback, set logging level to debug.
2026/03/07 15:01:24 WARNING mlflow.tracing.fluent: Failed to start span LangGraph: 'NonRecordingSpan' object has no attribute 'context'. For full traceback, set logging level to debug.


Evaluating:   0%|          | 0/3 [Elapsed: 00:00, Remaining: ?] 

Invalid type dict in attribute 'token' value sequence. Expected one of ['bool', 'str', 'bytes', 'int', 'float'] or None
Invalid type dict in attribute 'token' value sequence. Expected one of ['bool', 'str', 'bytes', 'int', 'float'] or None
Invalid type dict in attribute 'token' value sequence. Expected one of ['bool', 'str', 'bytes', 'int', 'float'] or None
Invalid type dict in attribute 'token' value sequence. Expected one of ['bool', 'str', 'bytes', 'int', 'float'] or None
Invalid type dict in attribute 'token' value sequence. Expected one of ['bool', 'str', 'bytes', 'int', 'float'] or None
Invalid type dict in attribute 'token' value sequence. Expected one of ['bool', 'str', 'bytes', 'int', 'float'] or None
Invalid type dict in attribute 'token' value sequence. Expected one of ['bool', 'str', 'bytes', 'int', 'float'] or None
Invalid type dict in attribute 'token' value sequence. Expected one of ['bool', 'str', 'bytes', 'int', 'float'] or None
Invalid type dict in attribute 'token' v

EvaluationResult(
  run_id: 8ef8baa3f7014a75b7c879089c029198
  metrics:
    custom_check/mean: 1.0
    correctness/mean: 0.3333333333333333
  result_df: 3 rows x 15 cols
)
